In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('../data/delivery_order.csv')

df.head()

,order_id,store_id,city,order_placed_time,order_delivered_time,delivery_minutes,distance_km,promised_sla_minutes,breached,day_of_week,hour_of_day
0,ORD-000001,STR-001,Jalandhar,2026-06-01 08:26:58,2026-06-01 08:37:11.200,10.22,0.81,10,True,Monday,8
1,ORD-000002,STR-001,Jalandhar,2026-06-01 15:51:56,2026-06-01 16:00:47.000,8.85,0.26,10,False,Monday,15
2,ORD-000003,STR-001,Jalandhar,2026-06-01 11:38:17,2026-06-01 11:49:51.200,11.57,3.77,10,True,Monday,11
3,ORD-000004,STR-001,Jalandhar,2026-06-01 10:19:27,2026-06-01 10:30:36.000,11.15,0.90,10,True,Monday,10
4,ORD-000005,STR-001,Jalandhar,2026-06-01 17:46:34,2026-06-01 17:56:09.400,9.59,3.08,10,False,Monday,17


In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 49944
Columns: 11


,order_id,store_id,city,order_placed_time,order_delivered_time,delivery_minutes,distance_km,promised_sla_minutes,breached,day_of_week,hour_of_day
0,ORD-000001,STR-001,Jalandhar,2026-06-01 08:26:58,2026-06-01 08:37:11.200,10.22,0.81,10,True,Monday,8
1,ORD-000002,STR-001,Jalandhar,2026-06-01 15:51:56,2026-06-01 16:00:47.000,8.85,0.26,10,False,Monday,15
2,ORD-000003,STR-001,Jalandhar,2026-06-01 11:38:17,2026-06-01 11:49:51.200,11.57,3.77,10,True,Monday,11
3,ORD-000004,STR-001,Jalandhar,2026-06-01 10:19:27,2026-06-01 10:30:36.000,11.15,0.90,10,True,Monday,10
4,ORD-000005,STR-001,Jalandhar,2026-06-01 17:46:34,2026-06-01 17:56:09.400,9.59,3.08,10,False,Monday,17


## Create the weekend feature 

In [4]:
df['is_weekend'] = df['day_of_week'].isin(
    ['Saturday', 'Sunday']
).astype(int)

df[['day_of_week', 'is_weekend']].head(10)

,day_of_week,is_weekend
0,Monday,0
1,Monday,0
2,Monday,0
3,Monday,0
4,Monday,0
5,Monday,0
6,Monday,0
7,Monday,0
8,Monday,0
9,Monday,0


In [5]:
# Prepare Features (X) and Target (y)
X = pd.get_dummies(
    df[['store_id', 'hour_of_day', 'is_weekend', 'distance_km']],
    columns=['store_id']
)

y = df['breached']

print("Features:", X.shape)
print("Target:", y.shape)

X.head()

Features: (49944, 15)
Target: (49944,)


,hour_of_day,is_weekend,distance_km,store_id_STR-001,store_id_STR-002,store_id_STR-003,store_id_STR-004,store_id_STR-005,store_id_STR-006,store_id_STR-007,store_id_STR-008,store_id_STR-009,store_id_STR-010,store_id_STR-011,store_id_STR-012
0,8,0,0.81,True,False,False,False,False,False,False,False,False,False,False,False
1,15,0,0.26,True,False,False,False,False,False,False,False,False,False,False,False
2,11,0,3.77,True,False,False,False,False,False,False,False,False,False,False,False
3,10,0,0.90,True,False,False,False,False,False,False,False,False,False,False,False
4,17,0,3.08,True,False,False,False,False,False,False,False,False,False,False,False


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (39955, 15)
Testing data: (9989, 15)


# Train the Random Forest Model

In [7]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed successfully.")

Model training completed successfully.


In [8]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.86      0.85      0.86      5870
        True       0.79      0.81      0.80      4119

    accuracy                           0.83      9989
   macro avg       0.83      0.83      0.83      9989
weighted avg       0.83      0.83      0.83      9989



In [9]:
importances = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances.head(10)

distance_km         0.359876
hour_of_day         0.216266
store_id_STR-002    0.120210
store_id_STR-003    0.095383
store_id_STR-001    0.044233
store_id_STR-009    0.034092
store_id_STR-010    0.033949
store_id_STR-011    0.028402
store_id_STR-012    0.023476
store_id_STR-006    0.010466
dtype: float64

In [10]:
importances = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances.head(10)

distance_km         0.359876
hour_of_day         0.216266
store_id_STR-002    0.120210
store_id_STR-003    0.095383
store_id_STR-001    0.044233
store_id_STR-009    0.034092
store_id_STR-010    0.033949
store_id_STR-011    0.028402
store_id_STR-012    0.023476
store_id_STR-006    0.010466
dtype: float64

# Save the Trained Model

In [11]:
import joblib

joblib.dump(model, '../data/breach_risk_model.pkl')

print("Model saved successfully.")

Model saved successfully.


In [12]:
model = joblib.load('../data/breach_risk_model.pkl')